In [1]:
import pandas as pd


bipad_flood = pd.read_csv(r"C:\Nepal_Flood_Project\Data\bipad_flood_nepal_2011-2026.csv", low_memory=False)
bipad_flood = bipad_flood[bipad_flood['Hazard'] == 'Flood']
flood_counts = bipad_flood['District'].value_counts()
top_flood_districts = flood_counts.head(10)


bipad_landslide = pd.read_csv(r"C:\Nepal_Flood_Project\Data\bipad_landslide_nepal_2011-2026.csv", low_memory=False)
bipad_landslide = bipad_landslide[bipad_landslide['Hazard'] == 'Landslide']
landslide_counts = bipad_landslide['District'].value_counts()
top_landslide_districts = landslide_counts.head(10)

print('Top 10 flood districts (by real incident count):')
print(top_flood_districts)
print('\nTop 10 landslide districts (by real incident count):')
print(top_landslide_districts)

priority_districts = sorted(set(top_flood_districts.index.tolist() + top_landslide_districts.index.tolist()))
print(f'\n{len(priority_districts)} unique priority districts:')
print(priority_districts)

Top 10 flood districts (by real incident count):
District
Jhapa        161
Morang       133
Sunsari       98
Kathmandu     91
Dang          82
Kailali       80
Udayapur      79
Rautahat      71
Panchthar     68
Kaski         67
Name: count, dtype: int64

Top 10 landslide districts (by real incident count):
District
Taplejung        224
Sankhuwasabha    198
Kaski            167
Rolpa            167
Palpa            151
Myagdi           150
Baglung          135
Sindhupalchok    134
Panchthar        134
Solukhumbu       133
Name: count, dtype: int64

18 unique priority districts:
['Baglung', 'Dang', 'Jhapa', 'Kailali', 'Kaski', 'Kathmandu', 'Morang', 'Myagdi', 'Palpa', 'Panchthar', 'Rautahat', 'Rolpa', 'Sankhuwasabha', 'Sindhupalchok', 'Solukhumbu', 'Sunsari', 'Taplejung', 'Udayapur']


In [2]:
bipad_all = pd.concat([bipad_flood, bipad_landslide], ignore_index=True)
priority_municipalities = (
    bipad_all[bipad_all['District'].isin(priority_districts)]
    [['District', 'Municipality']]
    .drop_duplicates()
    .sort_values(['District', 'Municipality'])
    .reset_index(drop=True)
)

print(f'Total municipalities found: {len(priority_municipalities)}')
print(priority_municipalities.to_string(index=False))

Total municipalities found: 173
     District            Municipality
      Baglung                 Badigad
      Baglung                 Baglung
      Baglung                  Bareng
      Baglung               Dhorpatan
      Baglung                  Galkot
      Baglung                 Jaimini
      Baglung             Kanthekhola
      Baglung               Nisikhola
      Baglung             Taman Khola
      Baglung              Tara Khola
         Dang                   Babai
         Dang             Banglachuli
         Dang             Dangisharan
         Dang                 Gadhawa
         Dang                 Ghorahi
         Dang                  Lamahi
         Dang                  Rajpur
         Dang                   Rapti
         Dang                Tulsipur
        Jhapa              Arjundhara
        Jhapa              Barhadashi
        Jhapa               Bhadrapur
        Jhapa               Birtamode
        Jhapa            Buddhashanti
        Jhapa     

In [3]:
import time
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="nepal_landslide_flood_project")
records = []
failed = []

for _, row in priority_municipalities.iterrows():
    district = row['District']
    muni = row['Municipality']
    query = f"{muni}, {district} District, Nepal"
    try:
        loc = geolocator.geocode(query, timeout=10)
        if loc:
            records.append({
                'district': district,
                'municipality': muni,
                'lat': loc.latitude,
                'lon': loc.longitude
            })
            print(f'{district} - {muni}: {loc.latitude}, {loc.longitude}')
        else:
            failed.append((district, muni))
            print(f'{district} - {muni}: NOT FOUND')
        time.sleep(1)
    except Exception as e:
        failed.append((district, muni))
        print(f'{district} - {muni}: FAILED - {e}')

municipalities_df = pd.DataFrame(records)
print(f'\nSuccessfully geocoded: {len(municipalities_df)} / {len(priority_municipalities)}')
print(f'Failed: {len(failed)}')

Baglung - Badigad: NOT FOUND
Baglung - Baglung: NOT FOUND
Baglung - Bareng: NOT FOUND
Baglung - Dhorpatan: NOT FOUND
Baglung - Galkot: NOT FOUND
Baglung - Jaimini: NOT FOUND
Baglung - Kanthekhola: NOT FOUND
Baglung - Nisikhola: NOT FOUND
Baglung - Taman Khola: NOT FOUND
Baglung - Tara Khola: NOT FOUND
Dang - Babai: NOT FOUND
Dang - Banglachuli: NOT FOUND
Dang - Dangisharan: NOT FOUND
Dang - Gadhawa: NOT FOUND
Dang - Ghorahi: NOT FOUND
Dang - Lamahi: NOT FOUND
Dang - Rajpur: NOT FOUND
Dang - Rapti: NOT FOUND
Dang - Tulsipur: NOT FOUND
Jhapa - Arjundhara: 26.7144365, 87.9604227
Jhapa - Barhadashi: 26.5282993, 87.9207548
Jhapa - Bhadrapur: 26.5455566, 88.0903762
Jhapa - Birtamode: 26.6430367, 87.9918647
Jhapa - Buddhashanti: 26.7398617, 88.0684095
Jhapa - Damak: 26.6650146, 87.7011507
Jhapa - Gauradhaha: NOT FOUND
Jhapa - Gauriganj: 26.4687423, 87.7261656
Jhapa - Haldibari: 26.5159384, 87.9994321
Jhapa - Jhapa: 26.5136308, 87.8662742
Jhapa - Kachankawal: 26.4156986, 88.0045459
Jhapa - Kam

In [5]:
import time
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="nepal_disaster_project_v2", timeout=15)
records = []
failed = []

for _, row in priority_municipalities.iterrows():
    district = row['District']
    muni = row['Municipality']
    query = f"{muni}, {district}, Nepal"
    try:
        loc = geolocator.geocode(query)
        if loc:
            records.append({'district': district, 'municipality': muni, 'lat': loc.latitude, 'lon': loc.longitude})
            print(f'{district} - {muni}: {loc.latitude}, {loc.longitude}')
        else:
            failed.append((district, muni))
            print(f'{district} - {muni}: NOT FOUND')
    except Exception as e:
        failed.append((district, muni))
        print(f'{district} - {muni}: FAILED - {e}')
    time.sleep(2)

municipalities_df = pd.DataFrame(records)
print(f'\nSuccessfully geocoded: {len(municipalities_df)} / {len(priority_municipalities)}')
print(f'Failed: {len(failed)}')

Baglung - Badigad: 28.2678013, 83.2509541
Baglung - Baglung: 28.2651055, 83.6029924
Baglung - Bareng: 28.1468175, 83.4691762
Baglung - Dhorpatan: 28.4902865, 83.0672741
Baglung - Galkot: 28.2505322, 83.3823394
Baglung - Jaimini: 28.1647289, 83.6014831
Baglung - Kanthekhola: 28.2631701, 83.5085249
Baglung - Nisikhola: 28.3763853, 83.0198922
Baglung - Taman Khola: 28.3569554, 83.1833073
Baglung - Tara Khola: NOT FOUND
Dang - Babai: 28.1803678, 82.0655029
Dang - Banglachuli: NOT FOUND
Dang - Dangisharan: 28.0855006, 82.154719
Dang - Gadhawa: 27.7597708, 82.6007196
Dang - Ghorahi: 28.0393605, 82.4866861
Dang - Lamahi: 27.9312024, 82.3410245
Dang - Rajpur: 27.8268287, 82.3555524
Dang - Rapti: 27.869703, 82.7170075
Dang - Tulsipur: 28.1312133, 82.2983065
Jhapa - Arjundhara: 26.7144365, 87.9604227
Jhapa - Barhadashi: 26.5282993, 87.9207548
Jhapa - Bhadrapur: 26.5455566, 88.0903762
Jhapa - Birtamode: 26.6430367, 87.9918647
Jhapa - Buddhashanti: 26.7398617, 88.0684095
Jhapa - Damak: 26.6650146,

In [6]:
RETRY_QUERIES = {
    ('Baglung', 'Tara Khola'):      'Tara Khola Rural Municipality, Baglung, Nepal',
    ('Dang', 'Banglachuli'):        'Banglachuli, Dang, Nepal',
    ('Jhapa', 'Gauradhaha'):        'Gauradaha, Jhapa, Nepal',
    ('Kathmandu', 'Kageshwori Manahora'): 'Kageshwori Manohara, Kathmandu, Nepal',
    ('Rolpa', 'Paribartan'):        'Paribartan Rural Municipality, Rolpa, Nepal',
    ('Sankhuwasabha', 'Savapokhari'): 'Sabhapokhari, Sankhuwasabha, Nepal',
    ('Sindhupalchok', 'Chautara Sangachokgadhi'): 'Chautara, Sindhupalchok, Nepal',
    ('Sunsari', 'Bhokraha Narsingh'): 'Bhokraha, Sunsari, Nepal',
    ('Sunsari', 'Devangunj'):       'Devanganj, Sunsari, Nepal',
}

retry_records = []
still_failed = []
for (district, muni), query in RETRY_QUERIES.items():
    try:
        loc = geolocator.geocode(query)
        if loc:
            retry_records.append({'district': district, 'municipality': muni, 'lat': loc.latitude, 'lon': loc.longitude})
            print(f'{district} - {muni}: {loc.latitude}, {loc.longitude}')
        else:
            still_failed.append((district, muni))
            print(f'{district} - {muni}: STILL NOT FOUND')
    except Exception as e:
        still_failed.append((district, muni))
        print(f'{district} - {muni}: FAILED - {e}')
    time.sleep(2)

retry_df = pd.DataFrame(retry_records)
print(f'\nRecovered: {len(retry_df)} / {len(RETRY_QUERIES)}')

Baglung - Tara Khola: STILL NOT FOUND
Dang - Banglachuli: STILL NOT FOUND
Jhapa - Gauradhaha: 26.564072, 87.715027
Kathmandu - Kageshwori Manahora: 27.7279083, 85.4084253
Rolpa - Paribartan: STILL NOT FOUND
Sankhuwasabha - Savapokhari: 27.4377287, 87.3560061
Sindhupalchok - Chautara Sangachokgadhi: 27.7513265, 85.7267833
Sunsari - Bhokraha Narsingh: 26.6069849, 87.125895
Sunsari - Devangunj: STILL NOT FOUND

Recovered: 5 / 9


In [7]:
manual_entries = pd.DataFrame([
    {'district': 'Baglung', 'municipality': 'Tara Khola', 'lat': 28.4200, 'lon': 83.1500},
    {'district': 'Dang', 'municipality': 'Banglachuli', 'lat': 28.0500, 'lon': 82.2200},
    {'district': 'Rolpa', 'municipality': 'Paribartan', 'lat': 28.4200, 'lon': 82.7500},
    {'district': 'Sunsari', 'municipality': 'Devangunj', 'lat': 26.5400, 'lon': 87.1600},
])

retry_df = pd.concat([retry_df, manual_entries], ignore_index=True)
print(f'Retry total now: {len(retry_df)} / 9')

Retry total now: 9 / 9


In [8]:
municipalities_df = pd.concat([municipalities_df, retry_df], ignore_index=True)
municipalities_df = municipalities_df.drop_duplicates(subset=['district','municipality'], keep='last').reset_index(drop=True)

print(f'Total municipalities geocoded: {len(municipalities_df)} / 173')

print('Latitude range:', municipalities_df['lat'].min(), 'to', municipalities_df['lat'].max())
print('Longitude range:', municipalities_df['lon'].min(), 'to', municipalities_df['lon'].max())

municipalities_df.to_csv(r'C:\Nepal_Flood_Project\Data\Districts_77\priority_municipalities_coords.csv', index=False)
print('Saved!')

Total municipalities geocoded: 173 / 173
Latitude range: 26.4096002 to 28.9281998
Longitude range: 80.55275 to 88.1274092
Saved!


In [9]:
district_terrain = pd.read_csv(r'C:\Nepal_Flood_Project\Data\Districts_77\nepal_70_districts_coords.csv').set_index('district')

DISTRICT_TO_TERRAIN_EXTRA = {
    'Kathmandu': 'Hilly', 'Kaski': 'Hilly', 'Morang': 'Terai',
}  

def get_terrain(district):
    if district in district_terrain.index:
        return district_terrain.loc[district, 'terrain']
    return DISTRICT_TO_TERRAIN_EXTRA.get(district)

def get_discharge_donor(district):
    if district in district_terrain.index:
        return district_terrain.loc[district, 'discharge_donor']
    return None  

import json
with open(r'C:\Nepal_Flood_Project\Data\Districts_77\slope_lookup_77.json') as f:
    district_slopes = json.load(f)

municipalities_df['terrain'] = municipalities_df['district'].apply(get_terrain)
municipalities_df['discharge_donor'] = municipalities_df['district'].apply(get_discharge_donor)
municipalities_df['district_slope'] = municipalities_df['district'].map(district_slopes)

print(municipalities_df[['district','municipality','terrain','discharge_donor','district_slope']].head(20))
print('\nMissing terrain:', municipalities_df['terrain'].isna().sum())
print('Missing discharge_donor:', municipalities_df['discharge_donor'].isna().sum())
print('Missing district_slope:', municipalities_df['district_slope'].isna().sum())

   district municipality terrain discharge_donor  district_slope
0   Baglung      Badigad   Hilly         pokhara            3.84
1   Baglung      Baglung   Hilly         pokhara            3.84
2   Baglung       Bareng   Hilly         pokhara            3.84
3   Baglung    Dhorpatan   Hilly         pokhara            3.84
4   Baglung       Galkot   Hilly         pokhara            3.84
5   Baglung      Jaimini   Hilly         pokhara            3.84
6   Baglung  Kanthekhola   Hilly         pokhara            3.84
7   Baglung    Nisikhola   Hilly         pokhara            3.84
8   Baglung  Taman Khola   Hilly         pokhara            3.84
9      Dang        Babai   Hilly         pokhara            1.04
10     Dang  Dangisharan   Hilly         pokhara            1.04
11     Dang      Gadhawa   Hilly         pokhara            1.04
12     Dang      Ghorahi   Hilly         pokhara            1.04
13     Dang       Lamahi   Hilly         pokhara            1.04
14     Dang       Rajpur 

In [10]:
DISTRICT_DONOR_FALLBACK = {
    'Kathmandu': 'kathmandu',
    'Kaski': 'pokhara',
    'Morang': 'biratnagar',
}

municipalities_df['discharge_donor'] = municipalities_df.apply(
    lambda r: DISTRICT_DONOR_FALLBACK.get(r['district'], r['discharge_donor'])
    if pd.isna(r['discharge_donor']) else r['discharge_donor'],
    axis=1
)

print('Missing discharge_donor now:', municipalities_df['discharge_donor'].isna().sum())
print(municipalities_df[municipalities_df['district'].isin(['Kathmandu','Kaski','Morang'])][['district','municipality','discharge_donor']].drop_duplicates(subset='district'))

Missing discharge_donor now: 0
     district    municipality discharge_donor
42      Kaski       Annapurna         pokhara
47  Kathmandu  Budhanilkantha       kathmandu
57     Morang         Belbari      biratnagar


In [11]:
municipalities_df.to_csv(r'C:\Nepal_Flood_Project\Data\Districts_77\priority_municipalities_full.csv', index=False)
print('Saved!')
print(municipalities_df.shape)

Saved!
(173, 7)


In [12]:
import os
os.makedirs(r'C:\Nepal_Flood_Project\Data\Districts_77\Municipalities', exist_ok=True)

In [13]:
import requests
import time

def download_nasa_power(lat, lon, name, start='20000101', end='20251231'):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        'parameters': 'T2M,PRECTOTCORR,RH2M,WS2M,GWETROOT',
        'community': 'AG', 'longitude': lon, 'latitude': lat,
        'start': start, 'end': end, 'format': 'CSV'
    }
    r = requests.get(url, params=params, timeout=60)
    fname = fr'C:\Nepal_Flood_Project\Data\Districts_77\Municipalities\nasa_{name}.csv'
    with open(fname, 'w') as f:
        f.write(r.text)
    return fname

downloaded, failed = [], []
for _, row in municipalities_df.iterrows():
    safe_name = f"{row['district']}_{row['municipality']}".replace(' ', '_')
    try:
        download_nasa_power(row['lat'], row['lon'], safe_name)
        downloaded.append(safe_name)
        print(f'Downloaded: {safe_name}')
    except Exception as e:
        failed.append(safe_name)
        print(f'FAILED: {safe_name} - {e}')
    time.sleep(1)

print(f'\nDownloaded: {len(downloaded)} / {len(municipalities_df)}')
if failed:
    print('Failed:', failed)

Downloaded: Baglung_Badigad
Downloaded: Baglung_Baglung
Downloaded: Baglung_Bareng
Downloaded: Baglung_Dhorpatan
Downloaded: Baglung_Galkot
Downloaded: Baglung_Jaimini
Downloaded: Baglung_Kanthekhola
Downloaded: Baglung_Nisikhola
Downloaded: Baglung_Taman_Khola
Downloaded: Dang_Babai
Downloaded: Dang_Dangisharan
Downloaded: Dang_Gadhawa
Downloaded: Dang_Ghorahi
Downloaded: Dang_Lamahi
Downloaded: Dang_Rajpur
Downloaded: Dang_Rapti
Downloaded: Dang_Tulsipur
Downloaded: Jhapa_Arjundhara
Downloaded: Jhapa_Barhadashi
Downloaded: Jhapa_Bhadrapur
Downloaded: Jhapa_Birtamode
Downloaded: Jhapa_Buddhashanti
Downloaded: Jhapa_Damak
Downloaded: Jhapa_Gauriganj
Downloaded: Jhapa_Haldibari
Downloaded: Jhapa_Jhapa
Downloaded: Jhapa_Kachankawal
Downloaded: Jhapa_Kamal
Downloaded: Jhapa_Kankai
Downloaded: Jhapa_Mechinagar
Downloaded: Jhapa_Shivasataxi
Downloaded: Kailali_Bardagoriya
Downloaded: Kailali_Bhajani
Downloaded: Kailali_Chure
Downloaded: Kailali_Dhangadhi
Downloaded: Kailali_Gauriganga
Downl

In [15]:
def read_nasa_file(filepath, name):
    skip_rows = 0
    with open(filepath, 'r') as f:
        for i, line in enumerate(f):
            if '-END HEADER-' in line:
                skip_rows = i + 1
                break
    df = pd.read_csv(filepath, skiprows=skip_rows)
    df.columns = df.columns.str.strip()
    df['location'] = name
    return df

In [16]:
test_df = read_nasa_file(
    r'C:\Nepal_Flood_Project\Data\Districts_77\Municipalities\nasa_Sindhupalchok_Melamchi.csv',
    'Sindhupalchok_Melamchi'
)
print(test_df.shape)
print(test_df.columns.tolist())
print(test_df.head())

(9497, 8)
['YEAR', 'DOY', 'T2M', 'PRECTOTCORR', 'RH2M', 'WS2M', 'GWETROOT', 'location']
   YEAR  DOY   T2M  PRECTOTCORR   RH2M  WS2M  GWETROOT                location
0  2000    1  1.23          0.0  26.04  1.56       0.4  Sindhupalchok_Melamchi
1  2000    2  2.20          0.0  25.27  1.57       0.4  Sindhupalchok_Melamchi
2  2000    3  2.48          0.0  24.48  1.47       0.4  Sindhupalchok_Melamchi
3  2000    4  2.51          0.0  23.86  1.63       0.4  Sindhupalchok_Melamchi
4  2000    5  2.76          0.0  19.45  1.58       0.4  Sindhupalchok_Melamchi


In [17]:
import os

all_muni_data = []
for _, row in municipalities_df.iterrows():
    safe_name = f"{row['district']}_{row['municipality']}".replace(' ', '_')
    filepath = fr'C:\Nepal_Flood_Project\Data\Districts_77\Municipalities\nasa_{safe_name}.csv'
    if os.path.exists(filepath):
        df_m = read_nasa_file(filepath, safe_name)
        df_m['district'] = row['district']
        df_m['municipality'] = row['municipality']
        df_m['terrain'] = row['terrain']
        df_m['discharge_donor'] = row['discharge_donor']
        df_m['slope'] = row['district_slope']
        all_muni_data.append(df_m)
    else:
        print(f'MISSING FILE: {safe_name}')

df_muni = pd.concat(all_muni_data, ignore_index=True)
print(f'Combined: {df_muni.shape[0]} rows, {df_muni["location"].nunique()} municipalities')

Combined: 1642981 rows, 173 municipalities


In [18]:
import numpy as np

df_muni['date'] = pd.to_datetime(
    df_muni['YEAR'].astype(str) + '-' + df_muni['DOY'].astype(str),
    format='%Y-%j'
)
df_muni['month'] = df_muni['date'].dt.month
df_muni['year']  = df_muni['date'].dt.year

def get_season(month):
    if month in [3, 4, 5]:   return 'pre_monsoon'
    elif month in [6, 7, 8, 9]: return 'monsoon'
    elif month in [10, 11]:  return 'post_monsoon'
    else:                    return 'winter'

df_muni['season'] = df_muni['month'].apply(get_season)

df_muni = df_muni.rename(columns={
    'T2M': 'temperature', 'PRECTOTCORR': 'rainfall',
    'RH2M': 'humidity', 'WS2M': 'wind_speed', 'GWETROOT': 'soil_moisture'
})
df_muni = df_muni.drop(columns=['YEAR', 'DOY'], errors='ignore')
df_muni = df_muni.replace(-999, np.nan)
df_muni = df_muni.sort_values(['location', 'date']).reset_index(drop=True)

# Rolling rainfall 
df_muni['rainfall_roll3'] = df_muni.groupby('location')['rainfall'].transform(lambda x: x.rolling(3, min_periods=1).sum())
df_muni['rainfall_roll7'] = df_muni.groupby('location')['rainfall'].transform(lambda x: x.rolling(7, min_periods=1).sum())

print('Date, season, and rolling rainfall done!')
print(df_muni[['location','date','temperature','rainfall','rainfall_roll7','soil_moisture','slope']].head())

Date, season, and rolling rainfall done!
          location       date  temperature  rainfall  rainfall_roll7  \
0  Baglung_Badigad 2000-01-01         2.25       0.0             0.0   
1  Baglung_Badigad 2000-01-02         3.25       0.0             0.0   
2  Baglung_Badigad 2000-01-03         2.99       0.0             0.0   
3  Baglung_Badigad 2000-01-04         3.48       0.0             0.0   
4  Baglung_Badigad 2000-01-05         3.20       0.0             0.0   

   soil_moisture  slope  
0           0.42   3.84  
1           0.42   3.84  
2           0.42   3.84  
3           0.42   3.84  
4           0.42   3.84  


In [19]:
import re

DHM_FOLDER = r"C:\Nepal_Flood_Project\Data\DHM Data"
STATION_CITY = {
    '550.05': 'kathmandu', '695': 'biratnagar', '280': 'nepalgunj',
    '430.5': 'pokhara', '795': 'illam', '439.7': 'manang',
}

def read_dhm_file(filepath):
    records = []
    with open(filepath, 'r', errors='ignore') as f:
        for line in f:
            line = line.strip()
            m = re.match(r'(\d{2}/\w+/\d{4}),(\S+)', line)
            if m:
                try:
                    date = pd.to_datetime(m.group(1), format='%d/%b/%Y').date()
                    discharge = float(m.group(2))
                    records.append({'date': date, 'discharge': discharge})
                except:
                    pass
    return pd.DataFrame(records)

discharge_lookup = {}
for station_no, city in STATION_CITY.items():
    filepath = os.path.join(DHM_FOLDER, f'DFL_{station_no}.txt')
    if os.path.exists(filepath):
        df_st = read_dhm_file(filepath)
        discharge_lookup[city] = df_st.set_index('date')['discharge'].to_dict()
        print(f'DHM loaded {city}: {len(df_st)} records')

DHM loaded kathmandu: 8605 records
DHM loaded biratnagar: 8525 records
DHM loaded nepalgunj: 8766 records
DHM loaded pokhara: 7305 records
DHM loaded illam: 5823 records
DHM loaded manang: 7011 records


In [20]:
discharge_records = []
for donor, vals in discharge_lookup.items():
    for date, value in vals.items():
        discharge_records.append({'discharge_donor': donor, 'date': date, 'discharge': value})

discharge_df = pd.DataFrame(discharge_records)
discharge_df['date'] = pd.to_datetime(discharge_df['date'])

df_muni['date'] = pd.to_datetime(df_muni['date'])
df_muni = df_muni.merge(discharge_df, on=['discharge_donor', 'date'], how='left')


donor_averages = {donor: np.mean(list(vals.values())) for donor, vals in discharge_lookup.items()}
fallback = df_muni['discharge_donor'].map(donor_averages)
df_muni['discharge'] = df_muni['discharge'].fillna(fallback)

print('Discharge feature added!')
print('Missing discharge:', df_muni['discharge'].isna().sum())
print(df_muni.groupby('discharge_donor')['discharge'].mean().round(1))

Discharge feature added!
Missing discharge: 0
discharge_donor
biratnagar    1679.3
illam           74.4
kathmandu       15.5
manang         226.1
nepalgunj     1347.2
pokhara         99.0
Name: discharge, dtype: float64


In [21]:
df_muni = df_muni.sort_values(['location', 'date']).reset_index(drop=True)
df_muni['discharge_roll3'] = df_muni.groupby('location')['discharge'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df_muni['discharge_roll7'] = df_muni.groupby('location')['discharge'].transform(lambda x: x.rolling(7, min_periods=1).mean())
print('Rolling discharge added!')

Rolling discharge added!


In [22]:
# --- Flood labels ---
bipad_flood_muni = bipad_flood.copy()
bipad_flood_muni['incident_date'] = pd.to_datetime(bipad_flood_muni['Incident on']).dt.date

flood_high_dates = {}
flood_medium_dates = {}
for loc in df_muni['location'].unique():
    district, muni = loc.split('_', 1)
    dates = set(bipad_flood_muni.loc[
        (bipad_flood_muni['District'] == district) & (bipad_flood_muni['Municipality'] == muni),
        'incident_date'
    ])
    flood_high_dates[loc] = dates
    med = set()
    for d in dates:
        for days_before in range(1, 8):
            pre = d - pd.Timedelta(days=days_before)
            if pre not in dates:
                med.add(pre)
    flood_medium_dates[loc] = med

def assign_flood_risk_muni(row):
    row_date = pd.Timestamp(row['date']).date()
    loc = row['location']
    terrain = row['terrain']
    if terrain == 'Mountain':
        return 'Low'
    if row_date in flood_high_dates.get(loc, set()):
        return 'High' if terrain == 'Terai' else 'Medium'
    if row_date in flood_medium_dates.get(loc, set()):
        return 'Medium'
    return 'Low'

df_muni['flood_risk'] = df_muni.apply(assign_flood_risk_muni, axis=1)
df_muni['flood_risk_label'] = df_muni['flood_risk'].map({'Low':0,'Medium':1,'High':2})

print('Flood labels done!')
print(df_muni.groupby('location')['flood_risk'].value_counts().unstack(fill_value=0).head(10))

Flood labels done!
flood_risk           High   Low  Medium
location                               
Baglung_Badigad         0  9489       8
Baglung_Baglung         0  9489       8
Baglung_Bareng          0  9489       8
Baglung_Dhorpatan       0  9422      75
Baglung_Galkot          0  9497       0
Baglung_Jaimini         0  9489       8
Baglung_Kanthekhola     0  9489       8
Baglung_Nisikhola       0  9489       8
Baglung_Taman_Khola     0  9497       0
Baglung_Tara_Khola      0  9497       0


In [23]:
bipad_landslide_muni = bipad_landslide.copy()
bipad_landslide_muni['incident_date'] = pd.to_datetime(bipad_landslide_muni['Incident on']).dt.date

ls_high_dates_muni = {}
ls_medium_dates_muni = {}
for loc in df_muni['location'].unique():
    district, muni = loc.split('_', 1)
    dates = set(bipad_landslide_muni.loc[
        (bipad_landslide_muni['District'] == district) & (bipad_landslide_muni['Municipality'] == muni),
        'incident_date'
    ])
    ls_high_dates_muni[loc] = dates
    med = set()
    for d in dates:
        for days_before in range(1, 15):
            pre = d - pd.Timedelta(days=days_before)
            if pre not in dates:
                med.add(pre)
    ls_medium_dates_muni[loc] = med

def assign_landslide_risk_muni(row):
    row_date = pd.Timestamp(row['date']).date()
    loc = row['location']
    terrain = row['terrain']
    slope = row['slope']
    if terrain == 'Terai':
        return 'Low'
    if row_date in ls_high_dates_muni.get(loc, set()):
        return 'High' if slope >= 10 else 'Medium'
    if row_date in ls_medium_dates_muni.get(loc, set()):
        return 'Medium' if slope >= 1 else 'Low'
    return 'Low'

df_muni['landslide_risk'] = df_muni.apply(assign_landslide_risk_muni, axis=1)
df_muni['landslide_risk_label'] = df_muni['landslide_risk'].map({'Low':0,'Medium':1,'High':2})

print('Landslide labels done!')
print('\nOverall flood label distribution:')
print(df_muni['flood_risk'].value_counts())
print('\nOverall landslide label distribution:')
print(df_muni['landslide_risk'].value_counts())

Landslide labels done!

Overall flood label distribution:
flood_risk
Low       1638253
Medium       4394
High          334
Name: count, dtype: int64

Overall landslide label distribution:
landslide_risk
Low       1630053
Medium      12548
High          380
Name: count, dtype: int64


In [24]:
OUTPUT_MUNI = r'C:\Nepal_Flood_Project\Data\Districts_77\final_municipality_dataset.csv'
df_muni.to_csv(OUTPUT_MUNI, index=False)
print(f'Saved: {OUTPUT_MUNI}')
print(f'Shape: {df_muni.shape[0]} rows x {df_muni.shape[1]} columns')

Saved: C:\Nepal_Flood_Project\Data\Districts_77\final_municipality_dataset.csv
Shape: 1642981 rows x 24 columns
